In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_log_error, make_scorer

In [2]:
df = pd.read_csv(r"C:\Users\bas41\Downloads\train kaggle.csv")
test = pd.read_csv(r"C:\Users\bas41\Downloads\test kaggle.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges,id
0,43,male,26.030,0,no,northeast,6837.36870,1
1,58,female,28.215,0,no,northwest,12224.35085,2
2,53,male,31.350,0,no,southeast,27346.04207,3
3,54,male,29.200,1,no,southwest,10436.09600,4
4,19,male,34.900,0,yes,southwest,34828.65400,5


In [3]:
def to_good_appearance(df: pd.DataFrame):
    df['smoker'] = np.int8(df['smoker'] == 'yes')
    
    df['male'] = np.int8(df['sex'] == 'male')
    
    df['northeast'] = np.int8(df['region'] == 'northeast')
    df['northwest'] = np.int8(df['region'] == 'northwest')
    df['southeast'] = np.int8(df['region'] == 'southeast')
    
    del df['sex']
    del df['region']
    del df['id']
    
    return df

df_v2 = to_good_appearance(df)
test_v2 = to_good_appearance(test)

df_v2

,age,bmi,children,smoker,charges,male,northeast,northwest,southeast
0,43,26.030,0,0,6837.36870,1,1,0,0
1,58,28.215,0,0,12224.35085,0,0,1,0
2,53,31.350,0,0,27346.04207,1,0,0,1
3,54,29.200,1,0,10436.09600,1,0,0,0
4,19,34.900,0,1,34828.65400,1,0,0,0
...,...,...,...,...,...,...,...,...,...
664,18,31.350,4,0,4561.18850,0,1,0,0
665,39,23.870,5,0,8582.30230,0,0,0,1
666,58,25.175,0,0,11931.12525,1,1,0,0
667,37,47.600,2,1,46113.51100,0,0,0,0


In [4]:
def add_features(df: pd.DataFrame):

    df['children'] = df['children'].where(df['children'] < 3, 3)

    df['age*bmi'] = df['age'] * df['bmi']
    df['age*children'] = df['age'] * df['children']
    df['bmi*children'] = df['bmi'] * df['children']
    df['smoker*bmi'] = df['smoker'] * df['bmi']
    df['age*smoker'] = df['age'] * df['smoker']
    df['male*bmi'] = df['male'] * df['bmi']
    df['age**2'] = df['age'] ** 2
    df['bmi**2'] = df['bmi'] ** 2
    df['children**2'] = df['children'] ** 2

    if 'charges' in df.columns:
        df['charges'] = np.log1p(df['charges'])

    feature_columns = [el for el in list(df.columns) if el != 'charges']
    
    return df, feature_columns

df_v3, feature_columns = add_features(df_v2)
test_v3, _ = add_features(test_v2)

df_v3

,age,bmi,children,smoker,charges,male,northeast,northwest,southeast,age*bmi,age*children,bmi*children,smoker*bmi,age*smoker,male*bmi,age**2,bmi**2,children**2
0,43,26.030,0,0,8.830304,1,1,0,0,1119.29,0,0.00,0.0,0,26.030,1849,677.560900,0
1,58,28.215,0,0,9.411267,0,0,1,0,1636.47,0,0.00,0.0,0,0.000,3364,796.086225,0
2,53,31.350,0,0,10.216364,1,0,0,1,1661.55,0,0.00,0.0,0,31.350,2809,982.822500,0
3,54,29.200,1,0,9.253122,1,0,0,0,1576.80,54,29.20,0.0,0,29.200,2916,852.640000,1
4,19,34.900,0,1,10.458224,1,0,0,0,663.10,0,0.00,34.9,19,34.900,361,1218.010000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664,18,31.350,3,0,8.425558,0,1,0,0,564.30,54,94.05,0.0,0,0.000,324,982.822500,9
665,39,23.870,3,0,9.057574,0,0,0,1,930.93,117,71.61,0.0,0,0.000,1521,569.776900,9
666,58,25.175,0,0,9.386990,1,1,0,0,1460.15,0,0.00,0.0,0,25.175,3364,633.780625,0
667,37,47.600,2,1,10.738883,0,0,0,0,1761.20,74,95.20,47.6,37,0.000,1369,2265.760000,4


In [5]:
x_train, x_test, y_train, y_test = train_test_split(df_v3.loc[:, feature_columns], df_v3.loc[:, 'charges'], test_size=0.2, random_state=42, shuffle=True)

def RMSLE(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

rmsle_scorer = make_scorer(
    RMSLE, 
    greater_is_better=False
)

In [6]:
grid_3 = {
    'n_estimators': range(90, 100),
    'max_depth': range(20, 40),
    'min_samples_leaf': [9],
    # 'regressor__max_features': np.linspace(0.1, 1, 2),
    # 'regressor__max_leaf_nodes': [None] + list(range(45, 50))
}

model = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=grid_3,
    cv=5,
    scoring=rmsle_scorer,
    n_jobs=-1,
    verbose=1
)

model.fit(x_train, y_train)

print("\n" + "="*50)
print("ЛУЧШИЕ ПАРАМЕТРЫ:")
print(model.best_params_)
print(f"Лучший CV score: {-model.best_score_:.4f}")

Fitting 5 folds for each of 200 candidates, totalling 1000 fits

ЛУЧШИЕ ПАРАМЕТРЫ Random Forest:
{'max_depth': 27, 'min_samples_leaf': 9, 'n_estimators': 99}
Лучший CV score (RMSLE): 0.0428


c:\Users\bas41\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\ma\core.py:2881: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


In [7]:
RFR = RandomForestRegressor(n_estimators=99, max_depth=24, min_samples_leaf=9, random_state=42)
RFR.fit(x_train, y_train)
y_pred = RFR.predict(x_test)
RMSLE(np.exp(y_test), np.exp(y_pred))

np.float64(0.32226420082624885)

In [8]:
y_pred = np.exp(RFR.predict(test_v2))

In [9]:
y_pred

array([ 8774.02375371,  4739.39919146, 26021.08978806,  9626.85366951,
       36447.17180329,  5671.34616445,  2078.04831262, 14536.02384236,
        4123.43410742, 12563.94575397, 19236.86887444,  8278.20590639,
        5907.64009085, 46974.07680749, 47072.18415382, 46852.16818822,
       12853.5568756 , 47006.30805389,  8095.48618237, 25298.00422217,
        4802.89363612,  9352.97916233,  1774.46751777,  3167.85510634,
       13001.06240164, 11630.03592594, 14525.42843716,  5809.5961349 ,
       11521.23836806,  1793.59961231,  8584.8226188 , 11854.04236825,
        2719.96605828,  5670.31487509,  4730.1040318 ,  7844.04193898,
        3294.07732707,  7435.46706258, 25587.17137817, 44994.61626768,
        4275.15965564,  4250.54226724, 12378.78063366, 13736.28911124,
        5845.76571803, 14089.84460144,  5082.60223542,  5106.39476281,
       46020.12755744,  5807.2327448 , 14558.89059387,  2450.1934633 ,
        6698.60813936,  2127.4234449 , 11078.990287  , 12248.28682918,
      

In [10]:
pred = pd.DataFrame(y_pred)
pred['id'] = range(770, 1439)
pred.columns = ['charges', 'id']
pred.to_csv(r"C:\Users\bas41\Downloads\answer.csv", index=False)